# 27 - Agent Observability & OpenTelemetry

## Scenario: Diagnosing Agent Failures in Production

When a traditional web server fails, you look at the stack trace. When an LLM Agent fails (or takes 45 seconds to respond), a stack trace won't help you. Was the LLM slow? Did a specific tool timeout? Did the agent get stuck in a thought loop and consume 20,000 tokens?

Standard flat logging (`print("doing stuff")`) is useless for agentic workflows. We need **Distributed Tracing**. 
In this notebook, we will use the standard **OpenTelemetry (OTEL)** framework to trace complex agent execution for Northstar Support, and analyze those traces to find bottlenecks and logic failures.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. Setting up OpenTelemetry

We configure OpenTelemetry to capture traces in memory so we can analyze them programmatically in this notebook. In production, these spans would be exported to Jaeger, Datadog, or Arize Phoenix.

In [2]:
# Note: You may need to pip install opentelemetry-api opentelemetry-sdk
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

# Configure the Tracer Provider
provider = TracerProvider()
memory_exporter = InMemorySpanExporter()
processor = SimpleSpanProcessor(memory_exporter)
provider.add_span_processor(processor)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("northstar.agent.tracer")
print("✅ OpenTelemetry In-Memory Tracer initialized.")

def analyze_traces():
    spans = memory_exporter.get_finished_spans()
    print("\n📊 --- OPENTELEMETRY SPAN ANALYSIS ---")
    for span in spans:
        duration_ms = (span.end_time - span.start_time) / 1e6
        attributes = dict(span.attributes) if span.attributes else {}
        
        print(f"🔹 Span: {span.name.ljust(25)} | Duration: {duration_ms:8.2f} ms")
        for key, val in attributes.items():
            print(f"      - {key}: {val}")
    print("--------------------------------------")


✅ OpenTelemetry In-Memory Tracer initialized.


## 2. Scenario A: The Latency Bottleneck

A customer complains that the Support Bot took 15 seconds to reply. Where did the time go? We will instrument the agent's tools and LLM calls with OTEL Spans.

In [3]:
import time
import random

def fetch_vpc_logs():
    """Simulates a very slow database query tool."""
    with tracer.start_as_current_span("tool.fetch_vpc_logs") as span:
        span.set_attribute("tool.name", "fetch_vpc_logs")
        
        # Simulate network latency
        time.sleep(1.5)
        
        span.set_attribute("tool.status", "success")
        return "Found 500 log lines."

def generate_llm_response(prompt: str):
    """Simulates the LLM call."""
    with tracer.start_as_current_span("llm.generate") as span:
        span.set_attribute("llm.model", "gpt-4o")
        
        # Simulate fast LLM inference
        time.sleep(0.2)
        return "The logs indicate a successful run."

def handle_slow_ticket(ticket: str):
    print(f"📩 Scenario A: Processing Ticket...")
    with tracer.start_as_current_span("agent.handle_ticket"):
        logs = fetch_vpc_logs()
        response = generate_llm_response(logs)
        print("✅ Scenario A Complete.")

memory_exporter.clear()
handle_slow_ticket("Why is my server slow?")
analyze_traces()
print("💡 INSIGHT A: The LLM is fast, but the internal DB is slow!")


📩 Scenario A: Processing Ticket...


✅ Scenario A Complete.

📊 --- OPENTELEMETRY SPAN ANALYSIS ---
🔹 Span: tool.fetch_vpc_logs       | Duration:  1505.13 ms
      - tool.name: fetch_vpc_logs
      - tool.status: success
🔹 Span: llm.generate              | Duration:   205.13 ms
      - llm.model: gpt-4o
🔹 Span: agent.handle_ticket       | Duration:  1710.82 ms
--------------------------------------
💡 INSIGHT A: The LLM is fast, but the internal DB is slow!


## 3. Scenario B: The Token Drain (Infinite Loop)

Agents can get stuck in "Thought Loops", invoking a tool over and over. We will log token usage as span attributes to alert on runaway agents.

In [4]:
def faulty_schema_tool():
    with tracer.start_as_current_span("tool.get_schema") as span:
        time.sleep(0.05)
        return "Error: Schema not found."

def handle_looping_ticket():
    print(f"\n📩 Scenario B: Looping Agent...")
    with tracer.start_as_current_span("agent.resolve_schema") as parent_span:
        total_tokens = 0
        for attempt in range(1, 4):
            with tracer.start_as_current_span(f"llm.reasoning.loop_{attempt}") as llm_span:
                tokens_used = 1500
                total_tokens += tokens_used
                llm_span.set_attribute("llm.tokens", tokens_used)
                time.sleep(0.1)
            faulty_schema_tool()
            
        parent_span.set_attribute("agent.total_tokens_consumed", total_tokens)
        print("❌ Agent gave up after 3 loops.")

memory_exporter.clear()
handle_looping_ticket()
analyze_traces()
print("💡 INSIGHT B: We can trigger billing alerts if `agent.total_tokens_consumed` spikes on a single parent span.")



📩 Scenario B: Looping Agent...


❌ Agent gave up after 3 loops.

📊 --- OPENTELEMETRY SPAN ANALYSIS ---
🔹 Span: llm.reasoning.loop_1      | Duration:   105.07 ms
      - llm.tokens: 1500
🔹 Span: tool.get_schema           | Duration:    55.09 ms
🔹 Span: llm.reasoning.loop_2      | Duration:   102.03 ms
      - llm.tokens: 1500
🔹 Span: tool.get_schema           | Duration:    51.46 ms
🔹 Span: llm.reasoning.loop_3      | Duration:   103.19 ms
      - llm.tokens: 1500
🔹 Span: tool.get_schema           | Duration:    50.23 ms
🔹 Span: agent.resolve_schema      | Duration:   468.12 ms
      - agent.total_tokens_consumed: 4500
--------------------------------------
💡 INSIGHT B: We can trigger billing alerts if `agent.total_tokens_consumed` spikes on a single parent span.


## 4. Scenario C: Exception & Error Tracing

If an agent's tool crashes entirely (e.g. division by zero, or a database connection dropped), OpenTelemetry captures the exception stack trace and attaches it to the span automatically, linking the Python crash directly to the Agent's specific reasoning step.

In [5]:
from opentelemetry.trace.status import Status, StatusCode

def risky_refund_tool():
    with tracer.start_as_current_span("tool.issue_refund") as span:
        time.sleep(0.1)
        try:
            # Simulate a crash in the tool code
            1 / 0
        except Exception as e:
            # Record the exception in the OTEL trace
            span.record_exception(e)
            # Mark the span status as ERROR so it glows RED in Datadog/Jaeger
            span.set_status(Status(StatusCode.ERROR, str(e)))
            raise

def handle_crashing_ticket():
    print(f"\n📩 Scenario C: Crashing Tool...")
    with tracer.start_as_current_span("agent.process_refund"):
        try:
            risky_refund_tool()
        except ZeroDivisionError:
            print("❌ Tool crashed violently.")

memory_exporter.clear()
handle_crashing_ticket()

# We manually extract the exception events from the span for demonstration
print("\n📊 --- OPENTELEMETRY ERROR ANALYSIS ---")
spans = memory_exporter.get_finished_spans()
for span in spans:
    if span.status.status_code == StatusCode.ERROR:
        print(f"🚨 CRITICAL FAILURE in span '{span.name}'")
        for event in span.events:
            if event.name == "exception":
                print(f"   Exception Type: {event.attributes.get('exception.type')}")
                print(f"   Message: {event.attributes.get('exception.message')}")

print("\n💡 INSIGHT C: OTEL automatically captures the traceback. When debugging, you instantly see WHICH tool crashed and WHY, mapped exactly to the agent's action.")



📩 Scenario C: Crashing Tool...
❌ Tool crashed violently.

📊 --- OPENTELEMETRY ERROR ANALYSIS ---
🚨 CRITICAL FAILURE in span 'tool.issue_refund'
   Exception Type: ZeroDivisionError
   Message: division by zero
   Exception Type: ZeroDivisionError
   Message: division by zero

💡 INSIGHT C: OTEL automatically captures the traceback. When debugging, you instantly see WHICH tool crashed and WHY, mapped exactly to the agent's action.


## 5. Scenario D: Parallel Sub-Agent Tracing

When you orchestrate multiple agents to work at the same time (e.g. searching 3 different databases concurrently), reading flat logs becomes impossible because the `print()` statements interleave randomly. 
Distributed tracing groups parallel spans neatly.

In [6]:
import threading

def parallel_search(database_name: str):
    with tracer.start_as_current_span(f"agent.search_{database_name}") as span:
        time.sleep(random.uniform(0.1, 0.4)) # Random delay
        span.set_attribute("db.name", database_name)

def handle_parallel_ticket():
    print(f"\n📩 Scenario D: Parallel Agents...")
    with tracer.start_as_current_span("orchestrator.gather_context"):
        threads = []
        for db in ["CustomerDB", "InventoryDB", "SupportKB"]:
            t = threading.Thread(target=parallel_search, args=(db,))
            threads.append(t)
            t.start()
        
        for t in threads:
            t.join()
    print("✅ Context gathered from all databases.")

memory_exporter.clear()
handle_parallel_ticket()
analyze_traces()

print("💡 INSIGHT D: Even though the threads execute simultaneously, the spans correctly attach to the `orchestrator.gather_context` parent. In a UI like Jaeger, this renders as a beautiful waterfall chart.")



📩 Scenario D: Parallel Agents...


✅ Context gathered from all databases.

📊 --- OPENTELEMETRY SPAN ANALYSIS ---
🔹 Span: agent.search_SupportKB    | Duration:   128.84 ms
      - db.name: SupportKB
🔹 Span: agent.search_CustomerDB   | Duration:   228.96 ms
      - db.name: CustomerDB
🔹 Span: agent.search_InventoryDB  | Duration:   249.12 ms
      - db.name: InventoryDB
🔹 Span: orchestrator.gather_context | Duration:   249.50 ms
--------------------------------------
💡 INSIGHT D: Even though the threads execute simultaneously, the spans correctly attach to the `orchestrator.gather_context` parent. In a UI like Jaeger, this renders as a beautiful waterfall chart.


## Checkpoint

**1. Why is tracing parallel agent execution vastly superior to using `print()` statements?**
- A) Print statements are illegal in Python 3.
- B) When running async/threaded agents, print statements interleave randomly on the console, making it impossible to read. OTEL traces inherently group parallel execution spans correctly under a parent span (waterfall graph).
- C) Print statements cost money.
- D) Traces generate training data for the LLM.
